In [ ]:
from asapdiscovery.data.services.fragalysis.fragalysis_reader import FragalysisFactory
from asapdiscovery.modeling.protein_prep import PreppedComplex
import pandas as pd
from pathlib import Path

In [ ]:
old_fragalysis_directory = "/data1/choderaj/paynea/asap-datasets/full_cross_dock_v2/mpro_fragalysis-04-01-24_curated"

In [ ]:
ff = FragalysisFactory(parent_dir=old_fragalysis_directory)

In [ ]:
plcs = ff.load()

In [ ]:
len(plcs)

In [ ]:
records = []
for c in plcs:
    records.append({"SMILES": c.ligand.smiles,
                    "Compound_Name": c.ligand.compound_name,
                    "Target_Name": c.target.target_name})

In [ ]:
df = pd.DataFrame.from_records(records)

In [ ]:
df.nunique()

## duplication problem:
### SMILES           498
### Compound_Name    515
### Target_Name      543

# Load Prepped

In [ ]:
prepped_path = Path("/data1/choderaj/paynea/asap-datasets/full_cross_dock/mpro_fragalysis-04-01-24_curated_cache_fixed")

In [ ]:
pcs = [PreppedComplex.from_json_file(f) for f in prepped_path.glob("./*/*.json")]

In [ ]:
records = []
for c in pcs:
    records.append({"SMILES": c.ligand.smiles,
                    "Compound_Name": c.ligand.compound_name,
                    "Target_Name": c.target.target_name})

In [ ]:
df = pd.DataFrame.from_records(records)

In [ ]:
df.nunique()

In [ ]:
smiles_counts = df.groupby("SMILES").nunique()

In [ ]:
smiles_counts[(smiles_counts["Compound_Name"] > 1)|(smiles_counts["Target_Name"] > 1)]

In [ ]:
smiles_counts[smiles_counts["Compound_Name"] != smiles_counts["Target_Name"]]

In [ ]:
smiles_counts[smiles_counts["Compound_Name"] > smiles_counts["Target_Name"]]

In [ ]:
smiles_counts[smiles_counts["Compound_Name"] < smiles_counts["Target_Name"]]

# Compound_Name

In [ ]:
cmpd_name_counts = df.groupby("Compound_Name").nunique()

In [ ]:
cmpd_name_counts[(cmpd_name_counts["SMILES"] > 1)|(cmpd_name_counts["Target_Name"] > 1)]

In [ ]:
cmpd_name_counts[cmpd_name_counts["SMILES"] > cmpd_name_counts["Target_Name"]]

In [ ]:
cmpd_name_counts[cmpd_name_counts["SMILES"] < cmpd_name_counts["Target_Name"]]

## conclusions
Assuming the SMILES is an accurate record.
For a given SMILES, sometimes it will have more than one compound name, and sometimes that compound name will have been crystallized multiple times

# SMILES with more than one Compound_Name

In [ ]:
smiles_with_multiple_names = smiles_counts[(smiles_counts["Compound_Name"] > 1)].index

In [ ]:
dup_smiles = smiles_counts[(smiles_counts["Compound_Name"] > 1)|(smiles_counts["Target_Name"] > 1)].index

In [ ]:
ordered_df = df[df.SMILES.isin(dup_smiles)].sort_values(['SMILES','Target_Name'])

In [ ]:
target_id_groups = []
for smiles in dup_smiles:
    target_id_groups.append(ordered_df[ordered_df["SMILES"] == smiles]["Target_Name"].tolist())

# How similar are duplicated ligand structures?

In [ ]:
from asapdiscovery.data.schema.ligand import Ligand
from asapdiscovery.data.backend.openeye import oechem
def calculate_ligand_rmsd(ref: Ligand, fit: Ligand) -> Ligand:
    fitmol = fit.to_oemol()
    refmol = ref.to_oemol()
    nConfs = fit.num_poses
    vecRmsd = oechem.OEDoubleArray(nConfs)
    success = oechem.OERMSD(refmol, fitmol, vecRmsd)
    if not success:
        print("RMSD calculation failed")
    return vecRmsd

In [ ]:
target_id_groups_flattened = [i for group in target_id_groups for i in group]

In [ ]:
pc_groups = [c for c in pcs if c.target.target_name in target_id_groups_flattened]

In [ ]:
name_to_complex = {c.target.target_name: c for c in pcs}

In [ ]:
pc_groups = [[name_to_complex[target_name] for target_name in group] for group in target_id_groups]

In [ ]:
def get_group_rmsds(pc_groups) -> pd.DataFrame:
    records = []
    for group in pc_groups:
        lig_combos = list(combinations(group, 2))
        for combo in lig_combos:
            pc1 = combo[0]
            pc2 = combo[1]
            rmsd = calculate_ligand_rmsd(pc1.ligand, pc2.ligand)
            records.append({"RMSD": rmsd[0],
                            "SMILES": pc1.ligand.smiles,
                            "Target_Name_1": pc1.target.target_name,
                            "Target_Name_2": pc2.target.target_name,
                            "Compound_Name_1": pc1.ligand.compound_name,
                            "Compound_Name_2": pc2.ligand.compound_name,})
    return pd.DataFrame.from_records(records)
            

In [ ]:
rmsd_df = get_group_rmsds(pc_groups)

In [ ]:
rmsd_df.to_csv("rmsds_for_duplicated_ligands.csv")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
rmsd_df.sort_values(["RMSD"], inplace=True)

In [ ]:
g = sns.scatterplot(rmsd_df, x="SMILES", y="RMSD")
plt.gca().set(xticklabels=[])
plt.savefig("rmsd_for_duplicated_ligands.png")
plt.savefig("rmsd_for_duplicated_ligands.svg")
plt.show()

In [ ]:
rmsd_df[rmsd_df["RMSD"] > 1]

# We should dedup them by date

In [ ]:
soaks_path = Path(old_fragalysis_directory) / "extra_files" / "Mpro_soaks.csv"

In [ ]:
soaks = pd.read_csv(soaks_path)

In [ ]:
from datetime import datetime

In [ ]:
def process_crystal_data(soaks):
    ddf = soaks.loc[:, ["Sample Name", "Data Collection Date"]]
    ddf["Sanitized_Date"] = ddf["Data Collection Date"].apply(date_processor)
    ddf.columns = ["Structure_Name", "Data_Collection_Date", "Structure_Date"]
    date_dict = ddf.set_index("Structure_Name").to_dict()["Structure_Date"]
    date_dict = {k: str(v) for k, v in date_dict.items() if str(v) != "NaT"}
    structure_to_cmpd_dict = {
        row["Sample Name"]: row["Compound ID"]
        for idx, row in soaks.iterrows()
        if row["Sample Name"] in date_dict
    }

    return date_dict, structure_to_cmpd_dict

In [ ]:
def date_processor(date_string):
    if type(date_string) == str and not date_string == "None":
        try:
            return datetime.strptime(date_string, "%Y-%m-%d %H:%M:%S")
        except ValueError:
            return datetime.strptime(date_string, "%d/%m/%Y %H:%M")
    else:
        return None

In [ ]:
date_dict, structure_to_cmpd_dict = process_crystal_data(soaks)

In [ ]:
ordered_df["Date"] = ordered_df.Target_Name.apply(lambda x: date_dict[x[:-3]])

In [ ]:
to_keep = ordered_df.sort_values("Date").groupby(["SMILES"]).head(1)

In [ ]:
targets_to_keep = set(to_keep.Target_Name.unique())

In [ ]:
all_duped_targets = set(ordered_df.Target_Name.unique())

In [ ]:
targets_to_remove = all_duped_targets - targets_to_keep